In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "5"
os.environ["OMP_NUM_THREADS"] = "16" # export OMP_NUM_THREADS=4
os.environ["OPENBLAS_NUM_THREADS"] = "16" # export OPENBLAS_NUM_THREADS=4 
os.environ["MKL_NUM_THREADS"] = "16" # export MKL_NUM_THREADS=6
os.environ["VECLIB_MAXIMUM_THREADS"] = "16" # export VECLIB_MAXIMUM_THREADS=4
os.environ["NUMEXPR_NUM_THREADS"] = "16" # export NUMEXPR_NUM_THREADS=6

In [2]:
%cd ../

/net/csefiles/coc-fung-cluster/Qianyu/stable_md/MatDeepLearn_dev


In [3]:
import json

with open('/net/csefiles/coc-fung-cluster/Qianyu/data/Li-O-P-Mn-Subset/data.json', 'r') as f:
    data = json.load(f)

In [4]:
import ase
from ase.cell import Cell
from ase.calculators.singlepoint import SinglePointCalculator

atoms_list = []
for d in data:
    # print(d['cell'])
    cell = Cell(d['cell'])
    atoms = ase.Atoms(
        symbols=d['atomic_numbers'],
        positions=d['positions'],
        cell=cell,
        pbc=[True, True, True]
    )
    # atoms.calc = SinglePointCalculator(
    #     energy=d['y'],
    #     forces=d['forces'],
    #     stress=d['stress'][0],
    #     atoms=atoms
    # )
    atoms.structure_id = d['structure_id']
    atoms_list.append(atoms)

In [5]:
import pandas as pd

# df = pd.read_csv('sim_results/mp_subset/test_traj/torchmd_4pairs_20aug_1_linspace_test_750K_NPT.csv')
# # df = df[df['min_min_dist'] < 1.0]
# df

In [6]:
to_sim = []
for id_ in ('mp-540415-4',):#df['structure_id'].iloc[:10]:
    to_sim.append([atoms for atoms in atoms_list if atoms.structure_id == str(id_)][0])

In [7]:
to_sim

[Atoms(symbols='Li4O28P8Mn4', pbc=True, cell=[[8.125330924987793, -8.730000445211772e-06, 1.233627438545227], [-4.5199999476608355e-06, 5.148319721221924, -6.4300002122763544e-06], [0.010924089699983597, -1.542000063636806e-05, 12.315476417541504]])]

In [8]:
from matdeeplearn.common.simulators import MetricMDSimulator

config_path = 'configs/sim_configs/mp_subset/torchmd/torchmd.yml'
simulator = MetricMDSimulator(config_path)

INFO:root:MDLCalculator instantiated from config: configs/calculator/mp_subset/torchmd/config_torchmd.yml
INFO:root:MDLCalculator: setting up torchmd_etEarly for calculation
/net/csefiles/coc-fung-cluster/Qianyu/.conda/envs/md_env/lib/python3.11/site-packages/torch_geometric/nn/conv/message_passing.py:1032: UserWarning: 'NeighborEmbedding.jittable' is deprecated and a no-op. Please remove its usage.
  warnings.warn(f"'{self.__class__.__name__}.jittable' is deprecated "
/net/csefiles/coc-fung-cluster/Qianyu/.conda/envs/md_env/lib/python3.11/site-packages/torch_geometric/nn/conv/message_passing.py:1032: UserWarning: 'EquivariantMultiHeadAttention.jittable' is deprecated and a no-op. Please remove its usage.
  warnings.warn(f"'{self.__class__.__name__}.jittable' is deprecated "
/net/csefiles/coc-fung-cluster/Qianyu/stable_md/MatDeepLearn_dev/matdeeplearn/common/ase_utils.py:210: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the

In [9]:
import random
import numpy as np
import torch

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

final_metrics = []
seed_everything(42)
for a in to_sim:
    metrics = simulator.run_simulation(a, save_traj=False)
    final_metrics.append(metrics)
    print(metrics)

{'structure_id': 'mp-540415-4', 'duration': 99.85734629631042}
